# Observabilidade do Amazon Bedrock AgentCore com LlamaIndex hospedado fora do AgentCore Runtime

Este notebook demonstra como configurar a observabilidade para um agente [LlamaIndex](https://docs.llamaindex.ai/en/stable/use_cases/agents/) hospedado fora do Amazon Bedrock AgentCore Runtime. Após concluir a configuração, você poderá visualizar o processo interno de tomada de decisão do agente LlamaIndex no painel de Observabilidade de IA Generativa no Amazon CloudWatch.

## O que você vai aprender
- Como configurar um agente LlamaIndex com a Instrumentação Python do Amazon OpenTelemetry
- Como visualizar e analisar traces do agente no Amazon CloudWatch GenAI Observability


## Pré-requisitos
- Habilitar a pesquisa transacional no Amazon CloudWatch. Usuários de primeira vez devem habilitar o CloudWatch Transaction Search para visualizar os spans e traces do Bedrock AgentCore. Para habilitar a pesquisa transacional, consulte nossa [documentação](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html).
- Log group e Log stream configurados no Amazon CloudWatch para serem adicionados às variáveis de ambiente.
- Conta AWS com acesso ao modelo Amazon Bedrock Claude Haiku 4.5 com Model ID: global.anthropic.claude-haiku-4-5-20251001-v1:0
- Credenciais AWS configuradas usando `aws configure` 
- Arquivo .env atualizado com as variáveis de ambiente. Um exemplo é fornecido em `.env.example`

## 1. Configuração e Instalação

Antes de executar este notebook, certifique-se de que já configurou seu ambiente virtual seguindo os passos abaixo:

1. Navegue até o diretório do LlamaIndex no seu terminal

2. Crie e ative um ambiente virtual:
   ```bash
   # Criar um ambiente virtual
   python -m venv venv

   # Ativar o ambiente virtual
   # No Windows
   venv\Scripts\activate
   # No macOS/Linux
   source venv/bin/activate
   ```

3. Instale as dependências:
   ```bash
   pip install -r requirements.txt
   ```

4. Ao abrir o notebook no Jupyter ou VS Code:
   - Selecione o kernel "venv" no seletor de kernel
   - Se o kernel não aparecer na lista, reinicie o Jupyter ou o VS Code

#### Implantando os pré-requisitos

Antes de começar, vamos criar um log group e um log stream para a observabilidade do AgentCore

In [ ]:
import boto3
cloudwatch_client = boto3.client("logs", region_name="us-west-2")
response = cloudwatch_client.create_log_group(
    logGroupName='agents/llama-index-agent-logs',
)
response

In [ ]:
response = cloudwatch_client.create_log_stream(
    logGroupName='agents/llama-index-agent-logs',
    logStreamName='default'
)
response

#### Habilitando a pesquisa transacional

Para executar este exemplo, você primeiro precisa habilitar a pesquisa transacional. Você pode fazer isso no console AWS seguindo este [link](https://console.aws.amazon.com/cloudwatch/home#xray:settings/transaction-search).

Uma vez nesta página, clique em editar e defina a opção para ingerir spans como logs estruturados no formato OpenTelemetry

![image.png](./images/transactional_search.png)
![image.png](./images/transactional_search2.png)

## 2. Configuração do Ambiente
Para habilitar a observabilidade para o seu agente LlamaIndex e enviar dados de telemetria para o Amazon CloudWatch, você precisará configurar as seguintes variáveis de ambiente. Utilizamos um arquivo `.env` para gerenciar essas configurações de forma segura, mantendo as credenciais sensíveis da AWS separadas do seu código, ao mesmo tempo que facilita a alternância entre diferentes ambientes.

**Certifique-se de que suas credenciais AWS estão configuradas**

Vamos criar um arquivo `.env` para configurar as variáveis de ambiente. Use `env.example` como modelo.

Variáveis de Ambiente Necessárias:

| Variável | Valor | Finalidade |
|----------|-------|-----------|
| `OTEL_PYTHON_DISTRO` | `aws_distro` | Usar a AWS Distro para OpenTelemetry (ADOT) |
| `OTEL_PYTHON_CONFIGURATOR` | `aws_configurator` | Definir o configurador AWS para o SDK ADOT |
| `OTEL_EXPORTER_OTLP_PROTOCOL` | `http/protobuf` | Configurar o protocolo de exportação |
| `OTEL_EXPORTER_OTLP_LOGS_HEADERS` | `x-aws-log-group=<SEU-LOG-GROUP>,x-aws-log-stream=<SEU-LOG-STREAM>,x-aws-metric-namespace=<SEU-NAMESPACE>` | Direcionar logs para grupos do CloudWatch |
| `OTEL_RESOURCE_ATTRIBUTES` | `service.name=<NOME-DO-SEU-AGENTE>` | Identificar seu agente nos dados de observabilidade |
| `AGENT_OBSERVABILITY_ENABLED` | `true` | Ativar o pipeline ADOT |
| `AWS_REGION` | `<SUA-REGIÃO>` | Região AWS |

In [ ]:
%%writefile .env
# AWS OpenTelemetry Distribution
OTEL_PYTHON_DISTRO=aws_distro
OTEL_PYTHON_CONFIGURATOR=aws_configurator

# Export Protocol
OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf
OTEL_TRACES_EXPORTER=otlp

# CloudWatch Integration (uncomment and configure as needed)
OTEL_EXPORTER_OTLP_LOGS_HEADERS=x-aws-log-group=agents/llama-index-agent-logs10,x-aws-log-stream=default,x-aws-metric-namespace=bedrock-agentcore

# Service Identification
OTEL_RESOURCE_ATTRIBUTES=service.name=agentic-llamaindex-agentcore
# Enable Agent Observability
AGENT_OBSERVABILITY_ENABLED=true

# Disable instrumentations to get rid of span noise (OPTIONAL)
OTEL_PYTHON_DISABLED_INSTRUMENTATIONS=jinja2

## 3. Carregar Variáveis de Ambiente

Vamos carregar as variáveis de ambiente do arquivo `.env`:

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Display the OTEL-related environment variables
otel_vars = [
    "OTEL_PYTHON_DISTRO",
    "OTEL_PYTHON_CONFIGURATOR",
    "OTEL_EXPORTER_OTLP_PROTOCOL",
    "OTEL_EXPORTER_OTLP_LOGS_HEADERS",
    "OTEL_RESOURCE_ATTRIBUTES",
    "AGENT_OBSERVABILITY_ENABLED",
    "OTEL_TRACES_EXPORTER",
    "OTEL_PYTHON_DISABLED_INSTRUMENTATIONS"
]

print("OpenTelemetry Configuration:")
for var in otel_vars:
    value = os.getenv(var)
    if value:
        print(f"{var}={value}")

## 4. Criar um Agente LlamaIndex em um arquivo Python

A implementação do agente aritmético LlamaIndex é fornecida em `llama_index_agent.py`. É um agente aritmético simples configurado com o modelo Claude 3 Haiku do Amazon Bedrock. A distribuição AWS OpenTelemetry irá lidar automaticamente com a configuração do provedor de traces ao usar o comando `opentelemetry-instrument`.

O agente é um agente aritmético simples que:

- Cria um FunctionAgent usando o modelo Claude Haiku do AWS Bedrock
- Define ferramentas aritméticas básicas para adição e multiplicação
- Atribui ao agente a tarefa de calcular uma expressão matemática simples: (121 + 2) * 5
- Executa o agente e retorna o resultado calculado

O Agente é configurado com o seguinte:

- Duas ferramentas de função aritmética: add e multiply
- Modelo Claude Haiku do Amazon Bedrock como seu Large Language Model
- Instrumentação OpenTelemetry para rastreamento e observabilidade

O agente é executado de forma assíncrona usando o método run do agente, que processa a consulta matemática e retorna o resultado.

In [ ]:
%%writefile llama_index_agent.py
###########################
#### Agent Code below: ####
###########################
import os
import asyncio
import logging
from llama_index.observability.otel import LlamaIndexOpenTelemetry
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent

# Initialize OpenTelemetry instrumentation for LlamaIndex
instrumentor = LlamaIndexOpenTelemetry(debug=True)

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configure LlamaIndex logging
logging.getLogger("llamaindex").setLevel(logging.INFO)

def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b


def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


def get_bedrock_model():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    
    try:
        # Let boto3 handle credential resolution automatically
        bedrock_model = BedrockConverse(
            model=model_id,
            region_name=region,
            # No explicit credentials - boto3 will find them automatically
        )
        logger.info(f"Successfully initialized Bedrock model: {model_id} in region: {region}")
        return bedrock_model
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock model: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

# Initialize the model
bedrock_model = get_bedrock_model()

# Create the arithmetic agent
agent = FunctionAgent(
    tools=[add, multiply],
    llm=bedrock_model,
)

# Start listening
instrumentor.start_registering()

# Execute the arithmetic task
query = """What is (121 + 2) * 5?"""

async def main():
    result = await agent.run(query)
    print("Result:", str(result))

asyncio.run(main())


## 5. AWS OpenTelemetry Python Distro

Agora que seu ambiente está configurado, vamos entender como a observabilidade funciona. A [AWS OpenTelemetry Python Distro](https://pypi.org/project/aws-opentelemetry-distro/) instrumenta automaticamente seu agente LlamaIndex para capturar dados de telemetria sem necessidade de alterações no código.

Esta distribuição fornece:
- **Auto-instrumentação** para seu Agente Strands hospedado fora do AgentCore Runtime (ou seja, EC2, Lambda etc.)
- **Configuração otimizada para AWS** para integração perfeita com o CloudWatch  

### Executando Seu Agente Instrumentado

Para capturar traces do seu agente LlamaIndex, use o comando `opentelemetry-instrument` em vez de executar o Python diretamente. Isso aplica automaticamente a instrumentação usando as variáveis de ambiente do seu arquivo `.env`:

```bash
opentelemetry-instrument python llama_index_agent.py
```

Este comando irá:

- Carregar sua configuração OTEL do arquivo .env
- Instrumentar automaticamente o LlamaIndex, chamadas ao Amazon Bedrock, ferramentas do agente e bancos de dados, e outras requisições feitas pelo agente
- Enviar traces para o CloudWatch
- Permitir que você visualize o processo de tomada de decisão do agente no painel de Observabilidade de IA Generativa

In [ ]:
!opentelemetry-instrument python llama_index_agent.py

## 6. Adicionando Rastreamento de Sessão

Para correlacionar traces entre múltiplas execuções do agente, você pode associar um ID de sessão aos seus dados de telemetria usando o baggage do OpenTelemetry:

```python
from opentelemetry import baggage, context
ctx = baggage.set_baggage("session.id", session_id)
```

Execute a versão habilitada com sessão:
```bash
opentelemetry-instrument python llama_indedx_agent_with_session.py --session-id "user-session-123"
```

## 7. Metadados Personalizados para Análise
Adicione atributos personalizados para habilitar filtragem, avaliações offline e análise de desempenho. Você precisaria modificar o código do seu agente para aceitar parâmetros adicionais:
```python
ctx = baggage.set_baggage("user.type", "premium")
ctx = baggage.set_baggage("experiment.id", "llama-agent")
ctx = baggage.set_baggage("conversation.topic", "arithmetic")
```

Exemplos de comandos com metadados personalizados:

```bash
# Testes A/B com diferentes experimentos
opentelemetry-instrument python agent.py --session-id "session-123" --experiment-id "model-a"
opentelemetry-instrument python agent.py --session-id "session-124" --experiment-id "model-b"

# Rastreando diferentes tipos de usuários
opentelemetry-instrument python agent.py --session-id "session-125" --user-type "premium"
opentelemetry-instrument python agent.py --session-id "session-126" --user-type "free"

# Execuções de avaliação offline
opentelemetry-instrument python agent.py --session-id "eval-001" --dataset "golden-set-v1"
```
Esses atributos aparecem nos traces do CloudWatch para filtragem e análise avançadas.

In [ ]:
%%writefile llama_index_agent_with_session.py
import os
import logging
import argparse
import asyncio
from opentelemetry import baggage, context
from llama_index.observability.otel import LlamaIndexOpenTelemetry
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent

def parse_arguments():
    parser = argparse.ArgumentParser(description='LlamaIndex Arithmetic Agent with Session Tracking')
    parser.add_argument('--session-id', 
                       type=str, 
                       required=True,
                       help='Session ID to associate with this agent run')
    return parser.parse_args()

def set_session_context(session_id):
    """Set the session ID in OpenTelemetry baggage for trace correlation"""
    ctx = baggage.set_baggage("session.id", session_id)
    token = context.attach(ctx)
    logging.info(f"Session ID '{session_id}' attached to telemetry context")
    return token

###########################
#### Agent Code below: ####
###########################

# Initialize OpenTelemetry instrumentation for LlamaIndex
instrumentor = LlamaIndexOpenTelemetry(debug=True)

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configure LlamaIndex logging
logging.getLogger("llamaindex").setLevel(logging.INFO)

def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b

def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b

def get_bedrock_model():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")

    try:
        # Let boto3 handle credential resolution automatically
        bedrock_model = BedrockConverse(
            model=model_id,
            region_name=region,
            # No explicit credentials - boto3 will find them automatically
        )
        logger.info(f"Successfully initialized Bedrock model: {model_id} in region: {region}")
        return bedrock_model
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock model: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

async def run_agent(query):
    # Initialize the model
    bedrock_model = get_bedrock_model()

    # Create the arithmetic agent
    agent = FunctionAgent(
        tools=[add, multiply],
        llm=bedrock_model,
    )

    # Start listening
    instrumentor.start_registering()

    # Execute the arithmetic task
    result = await agent.run(query)
    print("Result:", str(result))
    return result

def main():
    # Parse command line arguments
    args = parse_arguments()

    # Set session context for telemetry
    context_token = set_session_context(args.session_id)

    try:
        # Execute the arithmetic task
        query = """What is (121 + 2) * 5?"""

        # Run the async function in the event loop
        result = asyncio.run(run_agent(query))

    finally:
        # Detach context when done
        try:
            context.detach(context_token)
            logger.info(f"Session context for '{args.session_id}' detached")
        except ValueError as e:
            # Handle the context detachment error that might occur
            logger.error(f"Error detaching context: {str(e)}")

if __name__ == "__main__":
    main()



In [ ]:
!opentelemetry-instrument python llama_index_agent_with_session.py --session-id "session-1234"

## 8. Painel de Observabilidade de IA Generativa — Entendendo os Traces no AWS CloudWatch

Após a execução do seu agente LlamaIndex com instrumentação OpenTelemetry, você pode visualizar e analisar os traces no painel de Observabilidade de IA Generativa do AWS CloudWatch. Navegue até o Bedrock AgentCore e clique no Agente que você acabou de criar.

#### Página de Visualização de Sessões:

![llama_index_sessions.png](images/llama_index_sessions.png)


#### Página de Visualização de Traces:
Visualização de Traces:

![llama_index_sessions.png](images/llama_index_traces.png)


Detalhes do trace:

![llama_index_sessions.png](images/llama_index_trace_details.png)



## 9. Solução de Problemas

Se você não estiver vendo traces no Amazon CloudWatch ou X-Ray, verifique o seguinte:

1. **Credenciais AWS**: Certifique-se de que suas credenciais AWS estão configuradas corretamente
2. **Permissões IAM**: Verifique se seu usuário/role IAM possui permissões para o CloudWatch
3. **Região**: Confirme que você está verificando na região AWS correta
4. **Variáveis de Ambiente**: Verifique se todas as variáveis de ambiente OTEL_* estão definidas corretamente

## 10. Conclusão 

Parabéns! Você implementou e instrumentou um Agente LlamaIndex com o Modelo Amazon Bedrock que possui observabilidade através do Amazon CloudWatch.

- Agente aritmético LlamaIndex.
- Rastreamento completo com OpenTelemetry
- Traces para chamadas ao Amazon Bedrock, operações do LlamaIndex, etc.
- Nome do serviço: agentic-llamaindex-agentcore 

## 11. Próximos Passos

Agora que você configurou o LlamaIndex com OpenTelemetry, você pode:

1. **Adicionar Mais Agentes**: Criar arquiteturas multi-agente com diferentes padrões
2. **Adicionar Ferramentas ao seu agente**: Integrar ferramentas de busca, ferramentas de API ou ferramentas personalizadas
3. **[Configurar Alarmes](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/AlarmThatSendsEmail.html)**: Criar alarmes nas métricas que são importantes para o seu negócio como `latência`, `tokens de entrada` e `tokens de saída` etc.
